# KapInstruct-100M: Dataset Builder & Inspector

This notebook builds or inspects **KapInstruct-100M**, a reproducible 100,000,000-token instruction tuning dataset covering general instruction, math, coding, STEM reasoning, and code debugging.

In [ ]:
# 1. Install & Verify Dependencies
!pip install -q datasets transformers pyarrow pyyaml

import sys
import os
import glob
import json
import pyarrow as pa
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

print(f"Python {sys.version}")
print(f"PyTorch {torch.__version__}")

In [ ]:
# 2. Run Smoke Test & Pipeline Verification
!python scripts/04_kapinstruct_smoke_test.py

In [ ]:
# 3. PyTorch Arrow Shard Dataset for Training
class KapInstructDataset(Dataset):
    def __init__(self, shard_pattern="data/kapinstruct/*.arrow"):
        self.files = sorted(glob.glob(shard_pattern))
        if not self.files:
            raise FileNotFoundError(f"No shards found matching {shard_pattern}")
        
        self.index_map = []  # (file_idx, row_idx)
        self.tables = []
        for f_idx, f_path in enumerate(self.files):
            with open(f_path, "rb") as f:
                reader = pa.ipc.open_file(f)
                tbl = reader.read_all()
                self.tables.append(tbl)
                for row_idx in range(len(tbl)):
                    self.index_map.append((f_idx, row_idx))
        print(f"Loaded {len(self.index_map)} sequences across {len(self.files)} shards.")

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        f_idx, row_idx = self.index_map[idx]
        tbl = self.tables[f_idx]
        return {
            "input_ids": torch.tensor(tbl["input_ids"][row_idx].as_py(), dtype=torch.long),
            "labels": torch.tensor(tbl["labels"][row_idx].as_py(), dtype=torch.long),
            "attention_mask": torch.tensor(tbl["attention_mask"][row_idx].as_py(), dtype=torch.long),
        }

print("KapInstructDataset class defined successfully.")